In [1]:
# ============================================================
# STEP 2B — Load Cleaned CSVs into MySQL
# File: step2b_load_mysql.py
# Libraries used: pandas, mysql-connector-python
#
# Install (run once in terminal):
#   pip install mysql-connector-python openpyxl pandas
#
# Run AFTER:
#   step1_clean_data.py          (creates the 3 CSV files)
#   step2a_create_schema.sql     (run in MySQL Workbench first!)
# ============================================================

In [2]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


    pytz (>dev)
         ~^


In [3]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

In [4]:
# ── CONFIG — edit these to match your MySQL setup ─────────────
MYSQL_HOST     = "127.0.0.1"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = "shyam@8764"    # <-- CHANGE THIS
MYSQL_DATABASE = "netflix_db"
 

In [5]:
# ── CONNECT ──────────────────────────────────────────────────
print("Connecting to MySQL Workbench ...")
try:
    conn = mysql.connector.connect(
        host     = MYSQL_HOST,
        port     = MYSQL_PORT,
        user     = MYSQL_USER,
        password = MYSQL_PASSWORD,
        database = MYSQL_DATABASE
    )
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    ver = cursor.fetchone()
    print(f"  Connected! MySQL version: {ver[0]}")
except Error as e:
    print(f"  ERROR: {e}")
    print("  Check your MYSQL_PASSWORD and that MySQL Workbench server is running.")
    exit(1)

Connecting to MySQL Workbench ...
  Connected! MySQL version: 8.0.39


In [6]:
def bulk_insert(df, table_name):
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))

    sql = f"INSERT IGNORE INTO {table_name} ({cols}) VALUES ({placeholders})"

    data = []
    for row in df.itertuples(index=False):
        clean_row = []
        for v in row:
            if pd.isna(v):
                clean_row.append(None)
            else:
                clean_row.append(v)
        data.append(tuple(clean_row))

    cursor.executemany(sql, data) # insert one row
    conn.commit()
    return len(data)

In [7]:
# ── LOAD: netflix_titles ──────────────────────────────────────
print("\nLoading netflix_titles ...")
df = pd.read_csv("cleaned_netflix.csv", parse_dates=["date_added"])
# Select only the columns that exist in the table
cols_titles = ["show_id","type","title","director","cast","country",
               "date_added","year_added","month_added","month_name",
               "release_year","rating","audience_group","duration",
               "duration_minutes","duration_seasons","genres","description",
               "content_age_at_add"]
df_t = df[cols_titles].copy()
# Format date for MySQL
df_t["date_added"] = df_t["date_added"].dt.strftime("%Y-%m-%d")
n = bulk_insert(df_t,"netflix_titles")
print(f"  Inserted {n:,} rows into netflix_titles")



Loading netflix_titles ...
  Inserted 7,682 rows into netflix_titles


In [8]:
df_t.head()

,show_id,type,title,director,cast,country,date_added,year_added,month_added,month_name,release_year,rating,audience_group,duration,duration_minutes,duration_seasons,genres,description,content_age_at_add
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,8,August,2020,TV-MA,Adults,4,NaN,4.0,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,0
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2017,12,December,2008,TV-MA,Adults,143,143.0,NaN,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,9
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2019,1,January,2016,TV-PG,Family,124,124.0,NaN,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,3
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,3,March,2016,R,Adults,90,90.0,NaN,Comedies,New NFL star Thad buys his old teammates' belo...,0
4,s1001,TV Show,Blue Planet II,Unknown,David Attenborough,United Kingdom,2018-12-03,2018,12,December,2017,TV-G,Kids,1,NaN,1.0,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,1


In [9]:
# ── LOAD: netflix_genres ──────────────────────────────────────
print("\nLoading netflix_genres ...")
df_g = pd.read_csv("genres_exploded.csv")
cols_genres = ["show_id","type","title","release_year","year_added",
               "rating","audience_group","genre"]
df_g = df_g[cols_genres].copy()
n = bulk_insert(df_g, "netflix_genres")
print(f"  Inserted {n:,} rows into netflix_genres")



Loading netflix_genres ...
  Inserted 16,859 rows into netflix_genres


In [10]:
# ── LOAD: netflix_countries ───────────────────────────────────
print("\nLoading netflix_countries ...")
df_c = pd.read_csv("countries_exploded.csv")
cols_countries = ["show_id","type","title","release_year","year_added",
                  "rating","country_single"]
df_c = df_c[cols_countries].copy()
n = bulk_insert(df_c, "netflix_countries")
print(f"  Inserted {n:,} rows into netflix_countries")
 


Loading netflix_countries ...
  Inserted 8,954 rows into netflix_countries


In [11]:
# ── VERIFY ───────────────────────────────────────────────────
print("\n=== VERIFICATION ===")
for table in ["netflix_titles", "netflix_genres", "netflix_countries"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table:<25} : {count:,} rows") 
cursor.close()
conn.close()
print("\nAll data loaded! Now run step2c_create_views.sql in MySQL Workbench.")



=== VERIFICATION ===
  netflix_titles            : 7,682 rows
  netflix_genres            : 33,718 rows
  netflix_countries         : 26,862 rows

All data loaded! Now run step2c_create_views.sql in MySQL Workbench.
